# Project 14 — TulpStay Hotels: Cleaning + Gold Layer

**Format:** take-home · **Time-box: 60–90 min** (unit test included)

## Scenario

You are a data engineer at **TulpStay**, a small hotel chain with properties in 4 Dutch cities.
The booking export below comes straight from the front-desk tool and it is **dirty**.
Management wants a trustworthy silver table and three answers about revenue.

## Data dictionary

| column | target type | notes |
|---|---|---|
| booking_id | string | unique per booking |
| guest_first_name / guest_last_name | string | free text from the front desk |
| city | string | 4 cities |
| room_type | string | Standard / Deluxe / Suite |
| checkin_date | date | 2025-01 → 2025-06 |
| nights | int | length of stay |
| total_price | decimal(10,2) | EUR, total for the stay |
| discount_code | string | may be empty |

## Rules

- Upload `raw_data_14/hotel_bookings_14.csv` to Databricks yourself and read it (**strings first**).
- Every monthly result must use **yyyy-MM**.
- Questions marked **[W]** must be solved with a **window function**.
- Log every cleaning decision as a short comment where you make it.
- Time-box: stop at 90 min and note what is missing.

## The task (this is the whole assignment)

Profile the raw data, clean it, and write an idempotent silver table `hotel_bookings_silver`.
What counts as "clean" is your call — measure before you delete, and make sure the table has
one extra column: **`guest_full_name`** (proper casing, single space between first and last name).
Then answer the three gold questions below and finish with the bonus unit test.

In [ ]:
#FIRST READ THE DATA 
hotel_bookings_14_raw = (spark.read.format("csv")
                         .option("header",True)
                         .load("/Volumes/dev/spark_db/datasets/mini-projects/raw_data/hotel_bookings_14.csv")
)

hotel_bookings_14_raw.display()

In [ ]:
#NULLCHECK AND DISTINCT CHECK BECAUSE BOOKING_ID SHOULD BE UNIQUE

from pyspark.sql.functions import isnull, col
print(hotel_bookings_14_raw.filter(col("booking_id").isNull()).count())
print(hotel_bookings_14_raw.select("booking_id").distinct().count())
print(hotel_bookings_14_raw.select("booking_id").count())

In [ ]:

# FOUND 3 DUPLICATED ROWS AND CLEANED THEM

hotel_bookings_14_raw.groupBy("booking_id").count().filter(col("count")>1).display()

hotel_bookings_14_raw.filter(col("booking_id").isin("BK-2025","BK-2038","BK-2072")).display()

hotel_bookings_14_dedup = (
    hotel_bookings_14_raw.dropDuplicates()
)

hotel_bookings_14_dedup.filter(col("booking_id").isin("BK-2025","BK-2038","BK-2072")).display()


In [ ]:
# TYPE CHECKING

hotel_bookings_14_dedup.printSchema()

In [ ]:
#NORMALIZED COLUMNS (DELETING BLANKS, AND UNNECESSARY SYMBOLS, CAPITAL-LOWER LETTERS PROBLEM)

from pyspark.sql.functions import trim, lower, initcap,lit,replace

hotel_bookings_14_normalized = hotel_bookings_14_dedup.withColumns({
    "guest_first_name" : trim(initcap(col("guest_first_name"))),
    "guest_last_name"  : trim(initcap(col("guest_last_name"))),
    "city"             : trim(initcap(col("city"))),
    "room_type"        : trim(lower(col("room_type"))),
    "checkin_date"     : trim(col("checkin_date")),
    "nights"           : trim(col("nights")),
    "total_price"      : trim(replace(replace(replace(col("total_price"),lit("EUR"),lit("")),lit("€ "),lit("")),lit(","),lit(".")))
})

hotel_bookings_14_normalized.display()




In [ ]:
hotel_bookings_14_normalized.groupBy("room_type").count().display()      # meaningful
hotel_bookings_14_normalized.groupBy("city").count().display()           # meaningful
hotel_bookings_14_normalized.groupBy("discount_code").count().display()  # can be null, normal.


In [ ]:

print(hotel_bookings_14_normalized.filter(col("booking_id").isin("N/A","ERROR","UNKNOWN")).count())
print(hotel_bookings_14_normalized.filter(col("checkin_date").isin("N/A","ERROR","UNKNOWN")).count())
print(hotel_bookings_14_normalized.filter(col("nights").isin("N/A","ERROR","UNKNOWN")).count())
print(hotel_bookings_14_normalized.filter(col("total_price").isin("N/A","ERROR","UNKNOWN")).count())

# NIGHTS AND TOTAL_PRICE COLUMNS HAVE SOME PROBLEMATIC VALUES. WE ARE GOING TO MAKE THEM NULL BECAUSE THEY ARE PLACEHOLDERS.

In [ ]:
from pyspark.sql.functions import col, when

hotel_bookings_14_normalized = hotel_bookings_14_normalized.withColumns({
    "nights"        :  when(col("nights").isin("N/A","ERROR","UNKNOWN"),None).otherwise(col("nights")),
    "total_price"   :  when(col("total_price").isin("N/A","ERROR","UNKNOWN"),None).otherwise(col("total_price"))
     
})

hotel_bookings_14_normalized.filter(col("nights").isin("N/A","ERROR","UNKNOWN")).count()

# WE MADE THE PLACEHOLDERS NULL

In [ ]:
hotel_bookings_14_normalized.display()

In [ ]:
# CASTING DATA TYPES

from pyspark.sql.functions import concat_ws,try_to_date, coalesce

hotel_bookings_14_normalized = hotel_bookings_14_normalized.withColumns({
    "guest_full_name" : concat_ws(" ",col("guest_first_name"),col("guest_last_name")),
    "checkin_date"    : coalesce(
        try_to_date(col("checkin_date"),"yyyy-MM-dd"),
        try_to_date(col("checkin_date"),"MMM d, yyyy"),
        try_to_date(col("checkin_date"),"d/M/yyyy")
        ),
    "nights"          : col("nights").cast("long"),
    "total_price"     : col("total_price").cast("decimal(10,2)")
    
})

hotel_bookings_14_normalized.printSchema()
print(hotel_bookings_14_normalized.filter(col("checkin_date").isNull()).count())

hotel_bookings_14_normalized.display()



In [ ]:
# DECISION: negative nights are impossible stays -> dropped (2 rows, measured below);
# null nights/prices are KEPT (they were placeholders -> upstream DQ signal, not my data to invent)
hotel_booking_silver_df = hotel_bookings_14_normalized.filter((col("nights")>0) | (col("nights").isNull()))
hotel_booking_silver_df.display()


In [ ]:
hotel_booking_silver_df.filter(col("nights")<0).display()
hotel_booking_silver_df.filter(col("nights").isNull()).display()

In [ ]:
hotel_booking_silver_df.count()

In [ ]:
hotel_booking_silver_df.write.mode("overwrite").saveAsTable("dev.mini_projects.hotel_booking_silver_p14")

In [ ]:
spark.read.table("dev.mini_projects.hotel_booking_silver_p14").display()

## G1 — Monthly revenue per city

Total revenue per **city × month** (`yyyy-MM`). Which city+month combination generated
the most revenue?

In [ ]:
from pyspark.sql.functions import sum, date_format

monthly_revenue_df = hotel_booking_silver_df.withColumn("month", date_format("checkin_date", "yyyy-MM"))

monthly_revenue_df = monthly_revenue_df.groupBy("city", "month").agg(
    sum("total_price").alias("total_revenue")
).orderBy(col("total_revenue").desc())

monthly_revenue_df.display()

# AMSTERDAM + 2025-03 GENERATES THE MOST REVENUE (11,857.69).


## G2 [W] — 3-month moving average

Compute the company's total revenue per month, then add a **3-month moving average**
column (current month + the two previous months). Which month has the highest moving average?

*Hint you already own: this window needs an `order by` AND an explicit frame — think about
what the frame should say.*

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, date_format, avg

total_revenue_per_month_df = hotel_booking_silver_df.withColumn("month", date_format("checkin_date", "yyyy-MM"))
total_revenue_per_month_df = (
    total_revenue_per_month_df.groupBy("month").agg(
        sum("total_price").alias("total_revenue")
    )
)

# order by month + explicit frame: current row and the two previous months
w = Window.orderBy("month").rowsBetween(-2, Window.currentRow)

moving_avg_df = total_revenue_per_month_df.withColumn("moving_avg_revenue", avg("total_revenue").over(w))
moving_avg_df.display()

# 2025-05 HAS THE HIGHEST 3-MONTH MOVING AVERAGE (20,826.31).


## G3 [W] — Best city of each month

Rank the cities **within each month** by revenue (1 = best). Which city finished #1
in the most months?

*Careful: this partition/order choice is the mirror image of "top restaurants per city".*

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

each_month_by_revenue_df = hotel_booking_silver_df.withColumn("month", date_format("checkin_date", "yyyy-MM"))

# aggregate FIRST (city x month grain), THEN rank inside each month
best_city_each_month_df = each_month_by_revenue_df.groupBy("city", "month").agg(
    sum("total_price").alias("total_revenue")
)

w = Window.partitionBy("month").orderBy(col("total_revenue").desc())

best_city_each_month_df = best_city_each_month_df.withColumn("rank", rank().over(w))

best_city_each_month_df.filter(col("rank") == 1).display()

# ROTTERDAM IS #1 IN THE MOST MONTHS (2025-02, 2025-05, 2025-06 -> 3 of 6 months).


## BONUS — one easy unit test

Take your price-cleaning logic and put it in a file **`hotel_functions.py`** as:

```python
def clean_price(df):
    # input: df with a string column total_price ("€ 320.5", "EUR 410", "N/A", "320.5")
    # output: same df, total_price as double, junk values as null
    ...
```

Then create **`test_hotel_functions.py`** with ONE test, using this input (3 rows, by hand):

| booking_id | total_price |
|---|---|
| B1 | "€ 320.50" |
| B2 | "N/A" |
| B3 | "410.00" |

Write `expected_df` **by hand** (what should each price become?), assert with
`assertDataFrameEqual`, run pytest from a notebook cell with the `retcode` gate.


### The function and the test

The price-cleaning logic lives in [`hotel_functions.py`](hotel_functions.py) and the test in
[`test_hotel_functions.py`](test_hotel_functions.py), both next to this notebook.

The first version of `clean_price` had **no currency cleaning** — it only nulled the
placeholders and casted. The test caught it immediately:

```
NumberFormatException: [CAST_INVALID_INPUT] The value '€ 320.50' of the type "STRING"
cannot be cast to "DOUBLE" because it is malformed.
```

Two lessons from that red test:

- With **ANSI mode** on (Databricks default), `cast` does not silently null a malformed
  value — it throws. `try_cast` is the tolerant variant that returns null instead.
- The fix goes in the **function, not the test**: the expected values were already right.
  Clean first (`replace` the currency junk + `trim`), cast last.


In [ ]:
# hotel_functions.py  (saved next to this notebook)

from pyspark.sql.functions import col, when, lit, trim, replace

def clean_price(df):
    # placeholders -> null
    df = df.withColumns({
        "total_price": when(col("total_price").isin("N/A", "ERROR", "UNKNOWN"), None)
                       .otherwise(col("total_price"))
    })

    # clean first: strip currency junk, normalize decimal comma
    df = df.withColumns({
        "total_price": trim(replace(replace(replace(col("total_price"),
                       lit("EUR"), lit("")), lit("€"), lit("")), lit(","), lit(".")))
    })

    # cast last
    df = df.withColumns({
        "total_price": col("total_price").cast("double")
    })

    return df


In [ ]:
# test_hotel_functions.py  (saved next to this notebook)

import pytest
from pyspark.sql import SparkSession
from pyspark.testing import assertDataFrameEqual
from hotel_functions import clean_price

@pytest.fixture(scope="session")
def spark():
    return SparkSession.builder.getOrCreate()

def test_clean_price(spark):
    input_df = spark.createDataFrame([
        ("B1", "€ 320.50"),
        ("B2", "N/A"),
        ("B3", "410.00"),
    ], ["booking_id", "total_price"])

    # expected written BY HAND -- never copied from the function's own output
    expected_df = spark.createDataFrame([
        ("B1", 320.50),
        ("B2", None),
        ("B3", 410.00),
    ], ["booking_id", "total_price"])

    result_df = clean_price(input_df)
    assertDataFrameEqual(result_df, expected_df)


In [ ]:
#AUTORELOAD CELL

%reload_ext autoreload
%autoreload 2

In [ ]:
import pytest, sys
sys.dont_write_bytecode = True

retcode = pytest.main(["-v", "test_hotel_functions.py"])
assert retcode == 0, "Tests failed!"

# test_hotel_functions.py::test_clean_price PASSED -- 1 passed in 0.81s


## Defense questions (answered in English)

**1. G2: why does a moving average need BOTH an `order by` and an explicit `rows between` frame? What would you get with only the `order by`?**

The `order by` only defines the sequence of the rows — it doesn't say *which* rows to
average. The frame (`rows between 2 preceding and current row`) is what limits the window
to exactly three months. If I wrote only the `order by`, Spark applies its default frame —
*unbounded preceding to current row* — so I would silently get a **running (cumulative)
average** instead of a 3-month moving average. The numbers would look plausible, which is
what makes it dangerous: wrong logic, no error.

**2. G3: partitioned by month, ordered by revenue — project 13 did the opposite. How do you decide which column goes where?**

I read the question as a sentence: *"rank X **within each** Y"*. Whatever follows
"within each" is the `partition by`; the metric that decides the ranking is the
`order by`. Here it was "best city **within each month**" → partition by month, order by
revenue. In project 13 it was "top restaurants **within each city**" → partition by city.
The window syntax is identical — only the business sentence flips.

**3. Why must trim/initcap happen *before* concatenating `guest_full_name`?**

Because concatenation freezes the mess in. If I join `" anna "` and `"DE VRIES"` first,
I get one string where the original column boundaries are gone — fixing casing and inner
spacing afterwards is much harder (`initcap` on a merged string can't know which spaces
were padding). Cleaning each column first means `concat_ws(" ", ...)` joins two already
clean parts with exactly one space. General rule: **clean at the lowest grain, then combine.**

**4. The `clean_price` test covers three tiny rows. What does it prove — and what does it NOT prove about `hotel_bookings_silver`?**

It proves the **logic**: for the input shapes I chose (currency symbol, placeholder,
already-clean number), the function provably produces the right output — and it will
keep proving it on every rerun, so refactoring is safe. It does **not** prove anything
about the **data**: the real table could contain a shape I never put in the test
(a new currency, a negative price) and the test would still be green. Code quality is
the unit test's job; data quality is the job of the profiling checks and final checks
in the pipeline itself. You need both.


## Key Takeaways

- **Measure before you delete** — every drop is a number first: 143 raw → 140 after
  removing 3 exact duplicate rows (proven with `groupBy(booking_id).count()`), → 138 after
  dropping 2 impossible negative-nights rows. Null nights/prices stayed (upstream DQ signal).
- **Clean first, cast last** — `€ 1.234,50`-style strings go through `replace`/`trim`
  before `cast`; with ANSI mode on, a dirty string doesn't become a quiet null, it
  **throws** (`CAST_INVALID_INPUT`) — the unit test caught exactly that.
- **`concat_ws` after normalization** — `guest_full_name` built from already-trimmed,
  already-initcapped parts; cleaning after concatenation would be much harder.
- **Three date formats, one column** — `coalesce(try_to_date × 3)` parsed every
  `checkin_date` (0 nulls after, checked).
- **Window frames are not optional decoration** — `rows between 2 preceding and current
  row` is the difference between a 3-month moving average and an accidental running average.
- **partition by = "within each ..."** — the business sentence decides the window, not memory.
- **First red test that caught a real bug**: 3 hand-written rows exposed the missing
  currency cleaning in under a second. Fix the function, keep the expectation.
